In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

REPO_ROOT = Path.home() / "destriping-GLM-rebuttals"
sys.path.insert(0, str(REPO_ROOT))

from src.experiments_analysis.memory_requirements.peak_memory import build_memory_table

DATASETS = ["mouse_brain", "mouse_embryo", "human_lymph_node", "zebrafish_head"]
OUTPUT_DIR = REPO_ROOT / "results/my_notebooks/b2c_baseline_requirements"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
b2c_run_dirs: dict[str, Path] = {}
for dataset in DATASETS:
    cfg = yaml.safe_load(
        open(REPO_ROOT / f"experiments/benchmark_output_files/{dataset}.yaml")
    )
    run_dir = REPO_ROOT / cfg["not_factor_based_baseline"]["2SN"]
    b2c_run_dirs[dataset] = run_dir
    print(f"{dataset}: {run_dir}")


mouse_brain: /cluster/home/pmalsot/destriping-GLM-rebuttals/results/benchmark_destriping_model/mouse_brain_v2/b2c_destriping/2025-12-19/13-42-27/0__+dataset=mouse_brain
mouse_embryo: /cluster/home/pmalsot/destriping-GLM-rebuttals/results/benchmark_destriping_model/mouse_embryo/b2c_destriping/2026-01-09/11-52-17/0__+dataset=mouse_embryo
human_lymph_node: /cluster/home/pmalsot/destriping-GLM-rebuttals/results/benchmark_destriping_model/human_lymph_node/b2c_destriping/2026-01-10/00-33-05/0__+dataset=human_lymph_node
zebrafish_head: /cluster/home/pmalsot/destriping-GLM-rebuttals/results/benchmark_destriping_model/zebrafish_head/b2c_destriping/2026-01-10/00-24-43/0__+dataset=zebrafish_head


In [3]:
#overriding the mouse brain with the one which ran on the cluster -> memory can be read from there
b2c_run_dirs["mouse_brain"] = Path("results/benchmark_destriping_model/mouse_brain_v2/b2c_destriping/2026-03-18/17-10-08/0__+dataset=mouse_brain")

In [4]:
rows = []
for dataset, run_dir in b2c_run_dirs.items():
    time_dict = json.loads((run_dir / "time_dict.json").read_text())
    rows.append(
        {
            "dataset": dataset,
            "time_destripe_s": time_dict["time_destripe"],
            "time_destripe_min": round(time_dict["time_destripe"] / 60, 2),
        }
    )

df_time = pd.DataFrame(rows)
df_time.to_csv(OUTPUT_DIR / "time.csv", index=False)
df_time

,dataset,time_destripe_s,time_destripe_min
0,mouse_brain,4.478972,0.07
1,mouse_embryo,6.462791,0.11
2,human_lymph_node,1.991071,0.03
3,zebrafish_head,1.342126,0.02


## Peak memory

Must be run on the cluster (requires `sacct`).

Note: `.submitit/` lives in the **parent** of the run_dir (the Hydra multirun directory).

In [5]:
memory_runs = {dataset: run_dir.parent for dataset, run_dir in b2c_run_dirs.items()}
df_memory = build_memory_table(memory_runs)
df_memory.to_csv(OUTPUT_DIR / "peak_memory.csv", index=False)
df_memory

,run,peak_memory_GB
0,mouse_brain,2.61
1,mouse_embryo,3.66
2,human_lymph_node,0.89
3,zebrafish_head,1.19
